# AI-SPEAK — Training GRU & TCN Models

This notebook trains blendshape prediction models that map speech audio to 52 facial blendshape values at 60 FPS.

**Supported configurations:**
- `GRU + MFCC` — Fast training, good baseline
- `TCN + MFCC` — Fully causal, suitable for real-time
- `GRU + HuBERT` — More accurate, needs HuBERT precomputation
- `TCN + HuBERT` — Best accuracy + causal

**Runtime:** GPU (T4 or better recommended)

## 1. Setup

In [ ]:
# Running locally in VS Code — no Google Drive needed
# Results will be saved to a local folder (configured in Section 2)
print("Local environment — skipping Drive mount.")

In [ ]:
import os

# Set REPO_PATH to the root of your local clone
REPO_PATH = os.path.abspath(os.path.join(os.getcwd(), ".."))
os.chdir(REPO_PATH)

print("Working directory:", os.getcwd())

In [ ]:
# Install dependencies
!pip install -q librosa==0.10.1 soundfile==0.12.1 transformers==4.40.0 \
    pytorch-tcn gdown tqdm pandas scikit-learn matplotlib

## 2. Configuration

Set paths and choose which model to train.

In [ ]:
import os

# ============================================================
#  PATHS
# ============================================================
REPO_PATH    = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATA_ROOT    = os.path.join(REPO_PATH, "data")
RESULTS_ROOT = os.path.join(REPO_PATH, "results")
HUBERT_DIR   = os.path.join(REPO_PATH, "hubert_features")

# ============================================================
#  MODEL CHOICE
# ============================================================
# Options: "gru" | "tcn"
MODEL_TYPE = "gru"

# Options: "mfcc" | "hubert"
AUDIO_TYPE = "mfcc"

# ============================================================
#  RECOMMENDED TRAINING PARAMETERS
# ============================================================
EPOCHS           = 50
BATCH_SIZE       = 8      # reduce to 4 for HuBERT
LR               = 1e-3   # reduce to 5e-4 for HuBERT
WEIGHT_DECAY     = 1e-4
D_MODEL          = 256
PATIENCE         = 10
CHECKPOINT_EVERY = 5

VEL_LAM = 0.5
ACC_LAM = 0.1

SPEAKERS = "spk08 spk14"

# Set to "cuda" if you have a GPU, otherwise "cpu"
DEVICE = "cuda"

print(f"Model:  {MODEL_TYPE} + {AUDIO_TYPE}")
print(f"Data:   {DATA_ROOT}")
print(f"Output: {RESULTS_ROOT}")
print(f"Device: {DEVICE}")

## 3. Download Data

In [ ]:
import os
os.chdir(REPO_PATH)

if not os.path.exists(DATA_ROOT) or len(os.listdir(DATA_ROOT)) == 0:
    print("Downloading dataset...")
    !python download_data.py
else:
    print("Data already downloaded.")
    !ls {DATA_ROOT}

## 4. (Optional) Precompute HuBERT Features

Only needed when `AUDIO_TYPE = "hubert"`. Saves features to Drive so they can be reused across sessions.

In [ ]:
if AUDIO_TYPE == "hubert":
    import os
    os.makedirs(HUBERT_DIR, exist_ok=True)

    # Check if already computed
    existing = [f for f in os.listdir(HUBERT_DIR) if f.endswith('.npz')]
    if len(existing) > 0:
        print(f"Found {len(existing)} pre-computed HuBERT files in {HUBERT_DIR}")
    else:
        print("Precomputing HuBERT features... (this takes ~10-20 min)")
        !python scripts/precompute_hubert.py \
            --data_root {DATA_ROOT} \
            --output_dir {HUBERT_DIR} \
            --speakers spk08 spk14
        print("Done!")
else:
    print("Using MFCC features — HuBERT precomputation not needed.")

## 5. Train

Runs training with the parameters configured in Section 2.

In [ ]:
import os
os.chdir(REPO_PATH)

hubert_flag = f"--hubert_dir {HUBERT_DIR}" if AUDIO_TYPE == "hubert" else ""

cmd = f"""
python scripts/train.py \\
    --model        {MODEL_TYPE} \\
    --audio        {AUDIO_TYPE} \\
    --d_model      {D_MODEL} \\
    --epochs       {EPOCHS} \\
    --batch_size   {BATCH_SIZE} \\
    --lr           {LR} \\
    --weight_decay {WEIGHT_DECAY} \\
    --patience     {PATIENCE} \\
    --checkpoint_every {CHECKPOINT_EVERY} \\
    --vel_lam      {VEL_LAM} \\
    --acc_lam      {ACC_LAM} \\
    --data_root    {DATA_ROOT} \\
    --results_root {RESULTS_ROOT} \\
    --speakers     {SPEAKERS} \\
    --device       {DEVICE} \\
    {hubert_flag}
"""

print("Running:", cmd)
!{cmd}

## 6. Train All Configurations (GRU + TCN, MFCC only)

Run this cell to train both GRU and TCN with MFCC features sequentially.

In [ ]:
import os
os.chdir(REPO_PATH)

COMMON = f"""\
    --audio        mfcc \\
    --d_model      {D_MODEL} \\
    --epochs       {EPOCHS} \\
    --batch_size   8 \\
    --lr           1e-3 \\
    --weight_decay 1e-4 \\
    --patience     10 \\
    --checkpoint_every 5 \\
    --vel_lam      0.5 \\
    --acc_lam      0.1 \\
    --data_root    {DATA_ROOT} \\
    --results_root {RESULTS_ROOT} \\
    --speakers     spk08 spk14 \\
    --device       cuda"""

print("=" * 60)
print("Training GRU + MFCC")
print("=" * 60)
!python scripts/train.py --model gru {COMMON}

print("=" * 60)
print("Training TCN + MFCC")
print("=" * 60)
!python scripts/train.py --model tcn {COMMON}

## 7. View Results

In [ ]:
import os

print("Saved sessions:")
if os.path.exists(RESULTS_ROOT):
    for session in sorted(os.listdir(RESULTS_ROOT)):
        session_path = os.path.join(RESULTS_ROOT, session)
        ckpt_dir     = os.path.join(session_path, "checkpoints")
        best_model   = os.path.join(ckpt_dir, "best_model.pt")
        exists_mark  = "✓" if os.path.isfile(best_model) else "✗"
        print(f"  [{exists_mark}] {session}")
        # Print val_loss from best checkpoint
        if os.path.isfile(best_model):
            import torch
            ckpt = torch.load(best_model, map_location="cpu", weights_only=False)
            print(f"        best val_loss = {ckpt.get('val_loss', float('nan')):.4f}  "
                  f"epoch = {ckpt.get('epoch', '?')}")
else:
    print(f"Results root not found: {RESULTS_ROOT}")